Private learned route-teacher policy improvement.125 own/public features include exact retainedroute features, eightpairedfuturepile trials and uniform whole-game reservoir. CPU only. No event submission.

In [ ]:
import pathlib,subprocess,json,os
work=pathlib.Path("/kaggle/temp/triad-uno-route-counterfactual")
work.mkdir(parents=True,exist_ok=True)
sources={'uno_policy.py': '"""Linear action-ranking policy; inference uses filtered observations only."""\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nFEATURE_NAMES = [\n    "play", "number", "skip", "reverse", "draw_two", "wild", "wild_draw_four",\n    "remaining_colour_count", "remaining_colour_fraction", "remaining_match_count",\n    "attack_next_threat", "reverse_escape_threat", "wild_when_small_hand",\n    "finish", "keep", "choose_colour", "next_can_finish", "own_hand_size",\n]\nINITIAL_WEIGHTS = [8., .5, -.2, -.2, -.2, -2., -3., 2., 1., 1., 4., 2., 2., 12., -4., 0., 0., 0.]\n\n\ndef action_features(observation, action):\n    hand = observation["hand"]\n    by_ref = {card["card_ref"]: card for card in hand}\n    card = action.get("card") or by_ref.get(action.get("card_ref")) or {}\n    kind = card.get("kind")\n    play = action["type"] == "play"\n    colour = action.get("color") or card.get("color") or observation.get("active_color")\n    remaining = [c for c in hand if not (play and c["card_ref"] == action.get("card_ref"))]\n    colours = Counter(c.get("color") for c in remaining if c.get("color"))\n    matching = sum(c.get("color") == colour or c.get("kind") in ("wild", "wild_draw_four")\n                   or (kind == "number" and c.get("kind") == "number" and c.get("value") == card.get("value"))\n                   or (kind not in (None, "number", "wild", "wild_draw_four") and c.get("kind") == kind)\n                   for c in remaining)\n    sizes = observation.get("hand_sizes", [len(hand)] * 4)\n    seat, direction = observation["seat"], observation.get("direction", 1)\n    active = [s for s in range(len(sizes)) if s not in observation.get("finished_order", [])]\n    def neighbour(d):\n        for distance in range(1, len(sizes)):\n            other = (seat + distance * d) % len(sizes)\n            if other in active:\n                return max(1, sizes[other])\n        return max(1, len(hand))\n    next_size, previous_size = neighbour(direction), neighbour(-direction)\n    threat = 1. / next_size\n    attack = play and kind in ("skip", "draw_two", "wild_draw_four")\n    return [float(play), float(kind == "number"), float(kind == "skip"),\n            float(kind == "reverse"), float(kind == "draw_two"), float(kind == "wild"),\n            float(kind == "wild_draw_four"), min(colours[colour], 12) / 4.,\n            colours[colour] / max(1, len(remaining)), min(matching, 12) / 4.,\n            float(attack) * threat,\n            float(play and kind == "reverse") * (threat - 1. / previous_size),\n            float(play and kind in ("wild", "wild_draw_four")) / max(1, len(hand)),\n            float(play and len(hand) == 1), float(action["type"] == "keep"),\n            float(action["type"] == "choose_color"), float(play) * float(next_size <= 2),\n            float(play) * min(len(hand), 24) / 12.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES):\n            raise ValueError("wrong policy feature count")\n        self.weights = [float(w) for w in weights]\n\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        return max(legal, key=lambda action: sum(w * x for w, x in\n                   zip(self.weights, action_features(observation, action))))["action_id"]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-uno-linear-v1",\n            "features": FEATURE_NAMES, "weights": self.weights,\n            "training": metadata}, indent=2) + "\\n")\n', 'uno_policy_v2.py': '"""History-aware non-language UNO policy with public-information beliefs."""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom uno_policy import FEATURE_NAMES as V1_FEATURES, action_features as v1_features\n\nEXTRA_FEATURES = [\n    "successor_colour_absence", "successor_symbol_absence", "successor_colour_preference",\n    "successor_finish_risk", "remaining_colour_components", "largest_component_fraction",\n    "remaining_bridge_symbols", "remaining_duplicates", "remaining_colour_entropy",\n    "remaining_wilds", "chosen_colour_exhausted", "stack_penalty_play",\n    "successor_one_card_colour_absence", "successor_one_card_symbol_absence",\n]\nFEATURE_NAMES = V1_FEATURES + EXTRA_FEATURES\n\n\ndef symbol(card):\n    kind = card.get("kind")\n    if kind == "number":\n        return (kind, card.get("value"))\n    return (kind, None) if kind in ("skip", "reverse", "draw_two") else None\n\n\ndef public_beliefs(observation):\n    """Fold public events; a normal forced draw reveals pre-draw absences.\n\n    Unknown draws soften older evidence. Auto-kept or played drawn cards allow\n    exact post-draw absence deductions. Penalty draws do not imply any absence.\n    No simulator state, private opponent cards or seed is accessed here.\n    """\n    lack_colour, lack_symbol, preference, pending_draw = {}, {}, {}, {}\n    active_colour, top_symbol, pending_penalty = None, None, 0\n    for event in observation.get("history", []):\n        kind, seat = event.get("type"), event.get("seat")\n        if kind in ("game_start", "initial_discard"):\n            card = event.get("card", event.get("initial_discard", {})) or {}\n            active_colour = card.get("color", active_colour)\n            top_symbol = symbol(card)\n        elif kind == "initial_color_choice":\n            active_colour = event["color"]\n            preference[seat] = active_colour\n        elif kind in ("draw", "penalty_draw"):\n            count = event.get("drawn", 0) if kind == "draw" else event.get("count", 0)\n            for key in list(lack_colour):\n                if key[0] == seat:\n                    lack_colour[key] *= .75 ** count\n            for key in list(lack_symbol):\n                if key[0] == seat:\n                    lack_symbol[key] *= .92 ** count\n            if count:\n                preference.pop(seat, None)\n            if kind == "draw":\n                if active_colour:\n                    lack_colour[(seat, active_colour)] = .75 ** count\n                if top_symbol:\n                    lack_symbol[(seat, top_symbol)] = .92 ** count\n                pending_draw[seat] = (event.get("turn_number"), active_colour, top_symbol)\n            else:\n                pending_penalty = 0\n        elif kind in ("keep_auto", "play"):\n            card = event.get("card") or {}\n            if kind == "play":\n                colour, played_symbol = card.get("color"), symbol(card)\n                if colour:\n                    lack_colour[(seat, colour)] = 0.\n                if played_symbol:\n                    lack_symbol[(seat, played_symbol)] = 0.\n            drawn = pending_draw.get(seat)\n            if drawn and drawn[0] == event.get("turn_number") and (kind == "keep_auto" or event.get("from_draw")):\n                if drawn[1]:\n                    lack_colour[(seat, drawn[1])] = 1.\n                if drawn[2]:\n                    lack_symbol[(seat, drawn[2])] = 1.\n                pending_draw.pop(seat, None)\n            if kind == "play":\n                active_colour = event.get("active_color", card.get("color"))\n                top_symbol = symbol(card)\n                if card.get("kind") in ("wild", "wild_draw_four"):\n                    preference[seat] = active_colour\n                if "pending_penalty" in event:\n                    pending_penalty = event["pending_penalty"]\n    return {"colours": lack_colour, "symbols": lack_symbol, "preference": preference, "penalty": pending_penalty}\n\n\ndef structure(hand):\n    counts = Counter(c.get("color") for c in hand if c.get("color"))\n    groups = defaultdict(set)\n    for card in hand:\n        key = symbol(card)\n        if key and card.get("color"):\n            groups[key].add(card["color"])\n    adjacency = {colour: {colour} for colour in counts}\n    for colours in groups.values():\n        for colour in colours:\n            adjacency[colour].update(colours)\n    components, visited = [], set()\n    for colour in counts:\n        if colour in visited:\n            continue\n        frontier, connected = [colour], set()\n        while frontier:\n            current = frontier.pop()\n            if current not in connected:\n                connected.add(current)\n                frontier.extend(adjacency[current] - connected)\n        visited.update(connected)\n        components.append(sum(counts[c] for c in connected))\n    wilds = sum(c.get("kind") in ("wild", "wild_draw_four") for c in hand)\n    if wilds and components:\n        components = [sum(components)]\n    coloured = sum(counts.values())\n    entropy = -sum((n / coloured) * math.log(n / coloured) for n in counts.values()) if coloured else 0.\n    identical = Counter((c.get("color"), c.get("kind"), c.get("value")) for c in hand)\n    return [len(components) / 4., max(components, default=0) / max(1, len(hand)),\n            sum(len(colours) > 1 for colours in groups.values()) / 8.,\n            sum(n - 1 for n in identical.values()) / 4., entropy / 2., wilds / 4.]\n\n\ndef action_features(observation, action, beliefs=None):\n    beliefs = beliefs or public_beliefs(observation)\n    hand = observation["hand"]\n    play = action["type"] == "play"\n    card = action.get("card") or next((c for c in hand if c["card_ref"] == action.get("card_ref")), {})\n    kind = card.get("kind")\n    colour = action.get("color") or card.get("color") or observation.get("active_color")\n    remaining = [c for c in hand if not (play and c["card_ref"] == action.get("card_ref"))]\n    sizes, seat = observation["hand_sizes"], observation["seat"]\n    direction = observation.get("direction", 1)\n    active = [s for s in range(len(sizes)) if s not in observation.get("finished_order", [])]\n    if play and kind == "reverse":\n        direction = -direction\n    steps = 2 if play and kind in ("skip", "draw_two", "wild_draw_four") else 1\n    successor = seat\n    for _ in range(steps):\n        for distance in range(1, len(sizes) + 1):\n            other = (successor + distance * direction) % len(sizes)\n            if other in active:\n                successor = other\n                break\n    absence = beliefs["colours"].get((successor, colour), 0.)\n    absent_symbol = beliefs["symbols"].get((successor, symbol(card)), 0.)\n    threat = 1. / max(1, sizes[successor])\n    one_card = float(sizes[successor] == 1)\n    extras = [absence, absent_symbol, float(beliefs["preference"].get(successor) == colour),\n              threat * (1. - absence), *structure(remaining),\n              float(not any(c.get("color") == colour for c in remaining)),\n              float(play) * min(beliefs["penalty"], 24) / 12.,\n              one_card * absence, one_card * absent_symbol]\n    return v1_features(observation, action) + extras\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError("invalid v2 policy weights")\n        self.weights = list(weights)\n\n    def select(self, observation):\n        legal = observation["legal_actions"]\n        if len(legal) == 1:\n            return legal[0]["action_id"]\n        beliefs = public_beliefs(observation)\n        return max(legal, key=lambda action: sum(w * x for w, x in\n                   zip(self.weights, action_features(observation, action, beliefs))))["action_id"]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-uno-linear-v2", "features": FEATURE_NAMES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'uno_route_policy.py': '"""Own-hand sequence features for a frozen learned UNO base policy.\n\nThe bounded search imagines subsequent own plays with no intervening top-card\nchanges. It is a hand-structure feature, not a claim about future opponents.\nOnly filtered observations are read. No simulator or hidden state is imported.\n"""\nfrom collections import Counter\nfrom functools import lru_cache\nimport math\nfrom uno_policy import Policy as Base, action_features as base_features, FEATURE_NAMES as BASE_NAMES\nfrom uno_policy_v2 import public_beliefs, symbol\n\nEXTRA_NAMES = ["sequence_fraction", "sequence_stranded", "ending_wild",\n               "next_colour_absence", "next_symbol_absence", "next_colour_preference"]\nFEATURE_NAMES = BASE_NAMES + EXTRA_NAMES\n\n\ndef route_length(hand, colour, top, max_states=128, max_depth=8):\n    """Longest discovered normal-play path, with a fixed deterministic budget."""\n    cards = tuple(sorted(Counter((c.get(\'color\') or \'\', c[\'kind\'], c.get(\'value\') if c.get(\'value\') is not None else -1)\n                                 for c in hand).items()))\n    types = tuple(c for c, _ in cards)\n    counts = tuple(n for _, n in cards)\n    visits = 0\n\n    @lru_cache(maxsize=256)\n    def walk(counts, active, last_kind, last_value, depth):\n        nonlocal visits\n        if not depth or visits >= max_states:\n            return 0\n        visits += 1\n        best = 0\n        active_count = sum(n for n, (c, _, _) in zip(counts, types) if c == active)\n        for i, ((c, kind, value), n) in enumerate(zip(types, counts)):\n            if not n:\n                continue\n            wild = kind in (\'wild\', \'wild_draw_four\')\n            legal = (kind == \'wild\' or (kind == \'wild_draw_four\' and not active_count)\n                     or c == active or (kind == \'number\' and last_kind == \'number\' and value == last_value)\n                     or kind in (\'skip\', \'reverse\', \'draw_two\') and kind == last_kind)\n            if not legal:\n                continue\n            reduced = counts[:i] + (n - 1,) + counts[i + 1:]\n            options = sorted({other[0] for other, number in zip(types, reduced) if number and other[0]}) if wild else [c]\n            if not options:\n                options = [active]\n            for chosen in options:\n                length = 1 + walk(reduced, chosen, kind, value, depth - 1)\n                best = max(best, length)\n                if best >= min(depth, sum(counts)):\n                    return best\n        return best\n    return walk(counts, colour or \'\', top.get(\'kind\'), top.get(\'value\', -1), min(max_depth, len(hand)))\n\n\ndef action_features(observation, action, beliefs=None, cache=None):\n    beliefs = beliefs or public_beliefs(observation)\n    play = action[\'type\'] == \'play\'\n    card = action.get(\'card\') or next((c for c in observation[\'hand\'] if c[\'card_ref\'] == action.get(\'card_ref\')), {})\n    colour = action.get(\'color\') or card.get(\'color\') or observation.get(\'active_color\')\n    remaining = [c for c in observation[\'hand\'] if not(play and c[\'card_ref\'] == action.get(\'card_ref\'))]\n    top = card if play else observation.get(\'top_discard\', {})\n    key = (tuple(sorted((c.get(\'color\') or \'\', c[\'kind\'], c.get(\'value\', -1) if c.get(\'value\') is not None else -1)\n                        for c in remaining)), colour, top.get(\'kind\'), top.get(\'value\'))\n    if cache is not None and key in cache:\n        length = cache[key]\n    else:\n        length = route_length(remaining, colour, top)\n        if cache is not None:\n            cache[key] = length\n    sizes = observation[\'hand_sizes\']\n    direction = observation.get(\'direction\', 1)\n    if play and card.get(\'kind\') == \'reverse\':\n        direction = -direction\n    active = [s for s in range(len(sizes)) if s not in observation.get(\'finished_order\', [])]\n    successor = observation[\'seat\']\n    steps = 2 if play and card.get(\'kind\') in (\'skip\', \'draw_two\', \'wild_draw_four\') else 1\n    for _ in range(steps):\n        successor = next((successor + distance * direction) % len(sizes) for distance in range(1, len(sizes) + 1)\n                         if (successor + distance * direction) % len(sizes) in active)\n    denom = min(8, len(remaining))\n    extras = [length / max(1, denom), (denom - length) / 8.,\n              float(len(remaining) == 1 and remaining[0][\'kind\'] in (\'wild\', \'wild_draw_four\')),\n              beliefs[\'colours\'].get((successor, colour), 0.),\n              beliefs[\'symbols\'].get((successor, symbol(top)), 0.),\n              float(beliefs[\'preference\'].get(successor) == colour)]\n    return base_features(observation, action) + extras\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(w) for w in weights):\n            raise ValueError(\'invalid route policy weights\')\n        self.weights = list(weights)\n\n    def select(self, observation):\n        legal = observation[\'legal_actions\']\n        if len(legal) == 1:\n            return legal[0][\'action_id\']\n        beliefs, cache = public_beliefs(observation), {}\n        return max(legal, key=lambda a: sum(w * x for w, x in zip(self.weights, action_features(observation, a, beliefs, cache))))[\'action_id\']\n', 'uno_policy_features_v3.py': '"""Richer numeric own-hand encoding, canonicalized under colour permutations."""\nfrom collections import Counter\nimport math\nfrom uno_policy_v2 import FEATURE_NAMES as V2_FEATURES, action_features as v2_features, public_beliefs, symbol\n\nKINDS=[(\'number\',n) for n in range(10)]+[(\'skip\',None),(\'reverse\',None),(\'draw_two\',None)]\nFEATURE_NAMES=V2_FEATURES+[\n f\'remaining_colour_slot_{slot}_{kind}_{value}\' for slot in range(4) for kind,value in KINDS]+[\n \'remaining_wild_count\',\'remaining_wild_draw_four_count\']+[\n f\'action_number_{number}\' for number in range(10)]+[\n f\'opponent_slot_{slot}_{name}\' for slot in range(3) for name in (\'present\',\'hand_size\',\'one_card\',\'colour_absence\',\'symbol_absence\',\'colour_preference\')]+[\n \'selected_is_active_colour\',\'pending_penalty_size\',\'turn_progress\',\'active_opponents\',\'closest_opponent_risk\']\n\n\ndef action_features(observation,action,beliefs=None):\n beliefs=beliefs or public_beliefs(observation)\n hand=observation[\'hand\'];play=action[\'type\']==\'play\'\n card=action.get(\'card\') or next((c for c in hand if c[\'card_ref\']==action.get(\'card_ref\')), {})\n selected=action.get(\'color\') or card.get(\'color\') or observation.get(\'active_color\')\n remaining=[c for c in hand if not(play and c[\'card_ref\']==action.get(\'card_ref\'))]\n counts=Counter((c.get(\'color\'),c.get(\'kind\'),c.get(\'value\') if c.get(\'kind\')==\'number\' else None) for c in remaining)\n colours=[\'red\',\'yellow\',\'green\',\'blue\']\n order=[]\n for colour in (selected,observation.get(\'active_color\')):\n  if colour in colours and colour not in order:order.append(colour)\n def key(colour):\n  vector=tuple(counts[(colour,kind,value)] for kind,value in KINDS)\n  return sum(vector),vector\n order+=sorted((c for c in colours if c not in order),key=key,reverse=True)\n vector=[counts[(colour,kind,value)]/2. for colour in order for kind,value in KINDS]\n vector.extend(sum(c.get(\'kind\')==kind for c in remaining)/4. for kind in (\'wild\',\'wild_draw_four\'))\n vector.extend(float(card.get(\'kind\')==\'number\' and card.get(\'value\')==n) for n in range(10))\n seat=observation[\'seat\'];sizes=observation[\'hand_sizes\'];direction=observation.get(\'direction\',1)\n if play and card.get(\'kind\')==\'reverse\':direction=-direction\n finished=set(observation.get(\'finished_order\',[]))\n opponents=[(seat+i*direction)%len(sizes) for i in range(1,len(sizes)) if (seat+i*direction)%len(sizes) not in finished]\n for slot in range(3):\n  if slot>=len(opponents):vector.extend([0.]*6);continue\n  other=opponents[slot]\n  vector.extend([1.,sizes[other]/20.,float(sizes[other]==1),\n   beliefs[\'colours\'].get((other,selected),0.),beliefs[\'symbols\'].get((other,symbol(card)),0.),\n   float(beliefs[\'preference\'].get(other)==selected)])\n vector.extend([float(selected==observation.get(\'active_color\')),min(48,beliefs[\'penalty\'])/12.,\n  min(1000,observation.get(\'turn_number\',0))/1000.,len(opponents)/3.,\n  1./max(1,sizes[opponents[0]]) if opponents else 0.])\n return v2_features(observation,action,beliefs)+vector\n', 'uno_policy_features_v4.py': '"""Full own/public encoding plus exact retained hand-route feature values."""\nfrom uno_policy_features_v3 import FEATURE_NAMES as RICH_NAMES, action_features as rich_features\nfrom uno_route_policy import EXTRA_NAMES, action_features as route_features\n\nFEATURE_NAMES=RICH_NAMES+[\'route_\'+name for name in EXTRA_NAMES]\n\n\ndef action_features(observation,action,beliefs=None):\n    return rich_features(observation,action,beliefs)+route_features(observation,action,beliefs)[-len(EXTRA_NAMES):]\n', 'collect_uno_counterfactual.py': '"""Offline action-value targets from paired counterfactual SDK rollouts.\n\nHidden simulator state is used only for training rewards, as permitted by the\nSDK training guide. Exported policy inputs contain own/public information only.\nNo deployed policy sees seeds, hidden hands, or the simulator.\n"""\nimport argparse\nimport concurrent.futures\nimport copy\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\nfrom collections import Counter\nfrom uno_policy import Policy as V1, INITIAL_WEIGHTS\nfrom uno_policy_v2 import action_features, public_beliefs, FEATURE_NAMES\n\nROOT=Path(__file__).resolve().parent\nSDK=Path(os.environ.get(\'UNO_SDK\',ROOT.parent/\'unoSDK\'))\nsys.path.insert(0,str(SDK/\'local_evaluator/src\'))\nfrom triad_uno import engine\nfrom triad_uno.baselines import RandomAgent,HeuristicAgent\n\n\ndef actor(kind,seed,weights):\n    if kind==\'random\':return RandomAgent(seed)\n    if kind==\'heuristic\':return HeuristicAgent(seed)\n    if kind==\'initial\':return V1(INITIAL_WEIGHTS)\n    if len(weights)==24:\n        from uno_route_policy import Policy as RoutePolicy\n        return RoutePolicy(weights)\n    return V1(weights)\n\n\ndef reward(state,seat):\n    if seat in state.finished_order:\n        return state.config.position_points[state.finished_order.index(seat)]\n    remaining=state.config.position_points[len(state.outcome.placements):]\n    return sum(remaining)/len(state.outcome.unfinished)\n\n\ndef advance(state,actors):\n    seat=state.current_seat\n    obs=engine.observe(state,seat)\n    selected=actors[seat].select(obs)\n    action=engine.engine_action_from_id(state,seat,selected)\n    if action is None or not engine.step(state,action).ok:\n        raise RuntimeError(\'rollout actor produced an illegal action\')\n\n\ndef rollout(original,seat,selected,kinds,weights,seed,future_shuffle=False):\n    state=copy.deepcopy(original)\n    # Same trial seed across candidate actions yields common future random draws\n    # and random-opponent streams. Present cards are never altered/reallocated.\n    state.rng=random.Random(seed)\n    if future_shuffle:\n        # Future card order is unobserved. Each action shares the same trial\n        # permutation, while independent trials integrate its uncertainty.\n        # Existing hands are held fixed for these training-only rewards.\n        state.rng.shuffle(state.draw_pile)\n    actors=[actor(kind,seed*10+s,weights) for s,kind in enumerate(kinds)]\n    action=engine.engine_action_from_id(state,seat,selected)\n    if action is None or not engine.step(state,action).ok:\n        raise RuntimeError(\'counterfactual action is not legal at sampled state\')\n    while state.phase!=engine.FINISHED and seat not in state.finished_set:\n        advance(state,actors)\n    return reward(state,seat)\n\n\ndef collect_job(job):\n    seed,weights,samples,trials,max_actions,feature_version,*options=job\n    future_shuffle=options[0] if options else False\n    reservoir_sample=options[1] if len(options)>1 else False\n    feature_function=action_features\n    if feature_version==4:\n        from uno_policy_features_v4 import action_features as rich_features\n        feature_function=rich_features\n    elif feature_version==3:\n        from uno_policy_features_v3 import action_features as rich_features\n        feature_function=rich_features\n    rng=random.Random(seed^0x434F554E544552)\n    seat=seed%4\n    kinds=[rng.choice((\'incumbent\',\'initial\',\'heuristic\',\'random\')) for _ in range(4)]\n    kinds[seat]=\'incumbent\'\n    actors=[actor(kind,seed*10+s,weights) for s,kind in enumerate(kinds)]\n    state=engine.new_game(engine.GameConfig(max_turns=1000),seed)\n    rows=[];snapshots=[];eligible_count=0\n    def label(original,index):\n        obs=engine.observe(original,seat)\n        legal=list(obs[\'legal_actions\'])\n        if len(legal)>max_actions:\n            preferred=actor(\'incumbent\',seed,weights).select(obs)\n            kept=next(a for a in legal if a[\'action_id\']==preferred)\n            legal=[kept]+rng.sample([a for a in legal if a is not kept],max_actions-1)\n        beliefs=public_beliefs(obs)\n        features=[feature_function(obs,a,beliefs) for a in legal]\n        trial_values=[[rollout(original,seat,a[\'action_id\'],kinds,weights,seed*10_000_000+index*1000+trial,future_shuffle) for trial in range(trials)] for a in legal]\n        values=[sum(v)/trials for v in trial_values]\n        return (features,values,seed,trial_values,len(obs[\'hand\']),obs[\'phase\'])\n    while state.phase!=engine.FINISHED and seat not in state.finished_set:\n        current=state.current_seat\n        if current==seat and (reservoir_sample or len(rows)<samples):\n            obs=engine.observe(state,seat)\n            legal=list(obs[\'legal_actions\'])\n            if len(legal)>1:\n                if reservoir_sample:\n                    eligible_count+=1\n                    index=rng.randrange(eligible_count)\n                    if index<samples:\n                        saved=(copy.deepcopy(state),eligible_count)\n                        if len(snapshots)<samples:snapshots.append(saved)\n                        else:snapshots[index]=saved\n                elif rng.random()<.5:\n                    rows.append(label(state,len(rows)))\n        advance(state,actors)\n    if reservoir_sample:\n        rows=[label(original,index) for original,index in snapshots]\n    return rows\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\',type=Path,default=SDK/\'models/policy.json\')\n    parser.add_argument(\'--games\',type=int,default=1200)\n    parser.add_argument(\'--samples\',type=int,default=4)\n    parser.add_argument(\'--trials\',type=int,default=2)\n    parser.add_argument(\'--max-actions\',type=int,default=12)\n    parser.add_argument(\'--feature-version\',type=int,choices=(2,3,4),default=2)\n    parser.add_argument(\'--future-shuffle\',action=\'store_true\')\n    parser.add_argument(\'--reservoir-sample\',action=\'store_true\')\n    parser.add_argument(\'--workers\',type=int,default=4)\n    parser.add_argument(\'--seed-start\',type=int,default=10_000_000)\n    parser.add_argument(\'--output\',type=Path,default=ROOT/\'uno-counterfactual-data\')\n    args=parser.parse_args()\n    if min(args.games,args.samples,args.trials,args.workers)<1 or args.max_actions<2:parser.error(\'invalid counts\')\n    args.output.mkdir(parents=True,exist_ok=True)\n    document=json.loads(args.incumbent.read_text());weights=document[\'weights\']\n    features,targets,offsets,game_ids,trial_targets=[],[],[0],[],[]\n    hand_histogram,phase_histogram=Counter(),Counter()\n    started=time.time()\n    jobs=[(seed,weights,args.samples,args.trials,args.max_actions,args.feature_version,args.future_shuffle,args.reservoir_sample) for seed in range(args.seed_start,args.seed_start+args.games)]\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        for index,rows in enumerate(pool.map(collect_job,jobs)):\n            for vectors,values,seed,trial_values,hand_size,phase in rows:\n                features.extend(vectors);targets.extend(values);offsets.append(len(features));game_ids.append(seed)\n                trial_targets.extend(trial_values)\n                hand_histogram[hand_size]+=1;phase_histogram[phase]+=1\n            if (index+1)%100==0:print(json.dumps({\'games\':index+1,\'states\':len(game_ids),\'actions\':len(features),\'seconds\':round(time.time()-started,1)}),flush=True)\n    np.savez_compressed(args.output/\'counterfactual.npz\',features=np.asarray(features,dtype=np.float32),targets=np.asarray(targets,dtype=np.float32),offsets=np.asarray(offsets,dtype=np.int64),game_ids=np.asarray(game_ids,dtype=np.int64),trial_targets=np.asarray(trial_targets,dtype=np.float32))\n    names=FEATURE_NAMES\n    if args.feature_version==4:\n        from uno_policy_features_v4 import FEATURE_NAMES as rich_names\n        names=rich_names\n    elif args.feature_version==3:\n        from uno_policy_features_v3 import FEATURE_NAMES as rich_names\n        names=rich_names\n    metadata={\'method\':\'paired one-step policy-improvement rollouts\',\'input_features\':names,\'feature_version\':args.feature_version,\'games\':args.games,\'states\':len(game_ids),\'actions\':len(features),\'trials\':args.trials,\'future_shuffle\':args.future_shuffle,\'reward_conditioning\':\'present hidden simulator hands held fixed; future draw-pile permutations averaged\' if args.future_shuffle else \'original simulator card distribution\',\'max_actions\':args.max_actions,\'seed_start\':args.seed_start,\'sdk_commit\':\'0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe\',\'incumbent\':document,\'seconds\':round(time.time()-started,1)}\n    metadata.update(reservoir_sample=args.reservoir_sample,own_hand_size_histogram=dict(sorted(hand_histogram.items())),phase_histogram=dict(phase_histogram))\n    (args.output/\'metadata.json\').write_text(json.dumps(metadata,indent=2)+\'\\n\')\n    print(json.dumps({k:v for k,v in metadata.items() if k not in (\'incumbent\',\'input_features\')}),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in sources.items():(work/name).write_text(source)
incumbent={'format': 'triad-uno-route-linear-v1', 'features': ['play', 'number', 'skip', 'reverse', 'draw_two', 'wild', 'wild_draw_four', 'remaining_colour_count', 'remaining_colour_fraction', 'remaining_match_count', 'attack_next_threat', 'reverse_escape_threat', 'wild_when_small_hand', 'finish', 'keep', 'choose_colour', 'next_can_finish', 'own_hand_size', 'sequence_fraction', 'sequence_stranded', 'ending_wild', 'next_colour_absence', 'next_symbol_absence', 'next_colour_preference'], 'weights': [8.022466349142684, 0.44153069585311683, -0.06834035646341297, 1.4643534535685507, -2.338619866848944, -3.4540620544700498, -4.276279368110987, 1.6602353386949849, 1.0165100023007814, 0.901341302040876, 3.853185081203389, 2.4436903583872347, 1.7223969671761579, 11.872419584625783, -4.598739912392293, 3.347870880522802, 0.32778952616834095, -0.24548152213113314, 0.054801310646275025, -1.1349214815798592, -0.04225874297028254, 1.1191018681573717, 0.21891410472226475, -0.08122782541248305], 'training': {'method': 'frozen-base residual CEM, own-hand bounded sequence features', 'seed': 20261019, 'iterations': 24, 'sdk_commit': '0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe', 'selection_seed_start': 21000000, 'test_seed_start': 22000000, 'base': {'format': 'triad-uno-linear-v1', 'features': ['play', 'number', 'skip', 'reverse', 'draw_two', 'wild', 'wild_draw_four', 'remaining_colour_count', 'remaining_colour_fraction', 'remaining_match_count', 'attack_next_threat', 'reverse_escape_threat', 'wild_when_small_hand', 'finish', 'keep', 'choose_colour', 'next_can_finish', 'own_hand_size'], 'weights': [8.022466349142684, 0.44153069585311683, -0.06834035646341297, 1.4643534535685507, -2.338619866848944, -3.4540620544700498, -4.276279368110987, 1.6602353386949849, 1.0165100023007814, 0.901341302040876, 3.853185081203389, 2.4436903583872347, 1.7223969671761579, 11.872419584625783, -4.598739912392293, 3.347870880522802, 0.32778952616834095, -0.24548152213113314], 'training': {'method': 'cross-entropy policy optimization', 'seed': 20261014, 'generations': 30, 'population': 32, 'games_per_candidate': 64, 'validation': 3.005859375, 'sdk_commit': '0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe', 'feature_count': 18}}}}
(work/"incumbent.json").write_text(json.dumps(incumbent))
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/unoSDK.git",str(work/"unoSDK")],check=True)
subprocess.run(["git","checkout","--quiet","0cda3acc1bb5c05fa1803a8fbd7761770dc6d5fe"],cwd=work/"unoSDK",check=True)
os.environ["UNO_SDK"]=str(work/"unoSDK")
subprocess.run(["python",str(work/"collect_uno_counterfactual.py"),"--incumbent",str(work/"incumbent.json"),"--games","3000","--samples","4","--trials","8","--max-actions","8","--workers","4","--feature-version","4","--future-shuffle","--reservoir-sample","--seed-start","49000000","--output","/kaggle/working/output"],check=True,timeout=7200)
